# EPIC Clarity Drug Exposure Hydration

This notebook hydrates the OMOP DRUG_EXPOSURE table from EPIC Clarity medication order data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_MED` - Medication orders
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_MED_SIG` - Medication instructions/sig
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_MEDICATION` - Medication master reference

## OMOP Fields Populated
- drug_exposure_id (surrogate key)
- drug_source_value
- drug_exposure_start_date
- sig (patient instructions)
- visit_occurrence_id (if encounter available)

In [ ]:
%sql
-- TRUNCATE Gold table for Epic (run this to clear stale data before reload)
TRUNCATE TABLE _exponent.omop_epic.drug_exposure;

In [ ]:
%sql
-- Delete Epic records from Silver and Mapping tables (for full refresh)
DELETE FROM _exponent.omop_silver.drug_exposure WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_drug_exposure WHERE source_system = 'epic_clarity';

In [ ]:
%sql
-- Create silver_drug_exposure temp view for Epic Clarity
-- Includes name-based mapping to RxNorm Ingredient concepts
CREATE OR REPLACE TEMPORARY VIEW silver_drug_exposure AS
SELECT
  -- Drug concept - name-based mapping to RxNorm Ingredient
  COALESCE(drug_match.concept_id, 0) AS drug_concept_id,
  -- Start date
  DATE(COALESCE(om.START_DATE, om.ORDERING_DATE, om.ORDER_INST)) AS drug_exposure_start_date,
  COALESCE(om.START_DATE, om.ORDERING_DATE, om.ORDER_INST) AS drug_exposure_start_datetime,
  -- End date
  DATE(COALESCE(om.END_DATE, om.START_DATE, om.ORDERING_DATE, om.ORDER_INST)) AS drug_exposure_end_date,
  COALESCE(om.END_DATE, om.START_DATE, om.ORDERING_DATE, om.ORDER_INST) AS drug_exposure_end_datetime,
  -- Verbatim end date (only if explicit END_DATE)
  CASE WHEN om.END_DATE IS NOT NULL THEN DATE(om.END_DATE) ELSE NULL END AS verbatim_end_date,
  -- Type
  32817 AS drug_type_concept_id,
  -- Drug details
  NULL AS stop_reason,
  TRY_CAST(om.REFILLS AS INT) AS refills,
  TRY_CAST(om.QUANTITY AS DOUBLE) AS quantity,
  NULL AS days_supply,
  om.SIG AS sig,
  0 AS route_concept_id,
  NULL AS lot_number,
  -- Source values
  COALESCE(om.DISPLAY_NAME, om.DESCRIPTION, cm.NAME, cm.GENERIC_NAME) AS drug_source_value,
  0 AS drug_source_concept_id,
  CAST(om.MED_ROUTE_C AS STRING) AS route_source_value,
  om.DOSAGE AS dose_unit_source_value,
  -- FK source values
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', om.PAT_ID) AS person_source_value,
  CASE
    WHEN om.MED_PRESC_PROV_ID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'epic_clarity', 'clarity_ser', 'PROV_ID', om.MED_PRESC_PROV_ID)
    ELSE NULL
  END AS provider_source_value,
  CASE
    WHEN om.PAT_ENC_CSN_ID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'epic_clarity', 'pat_enc', 'PAT_ENC_CSN_ID', CAST(om.PAT_ENC_CSN_ID AS STRING))
    ELSE NULL
  END AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'order_med', 'ORDER_MED_ID', CAST(om.ORDER_MED_ID AS STRING)) AS drug_exposure_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.order_med om
LEFT JOIN _exponent._bronze_epic_clarity.clarity_medication cm
  ON cm.MEDICATION_ID = om.MEDICATION_ID
-- Name-based mapping to RxNorm Ingredient (first word of generic name)
LEFT JOIN _exponent.omop.concept drug_match
  ON drug_match.vocabulary_id = 'RxNorm'
  AND drug_match.concept_class_id = 'Ingredient'
  AND drug_match.standard_concept = 'S'
  AND LOWER(drug_match.concept_name) = LOWER(TRIM(SPLIT(cm.GENERIC_NAME, ' ')[0]))
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', om.PAT_ID)
  AND stp.active_flag = TRUE
WHERE om.ORDER_MED_ID IS NOT NULL
  AND om.PAT_ID IS NOT NULL
  AND COALESCE(om.START_DATE, om.ORDERING_DATE, om.ORDER_INST) IS NOT NULL
  AND (om.DELETE_FLAG = 0 OR om.DELETE_FLAG IS NULL)
  -- Exclude implausible dates before 1950
  AND DATE(COALESCE(om.START_DATE, om.ORDERING_DATE, om.ORDER_INST)) >= '1950-01-01'

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_drug_exposure LIMIT 10

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING (
  SELECT * FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY drug_exposure_source_value, drug_type_concept_id
        ORDER BY drug_exposure_start_date DESC
      ) AS rn
    FROM silver_drug_exposure
  ) WHERE rn = 1
) AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value
   AND t.drug_type_concept_id = s.drug_type_concept_id

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id             = s.drug_concept_id,
  t.drug_exposure_start_date    = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date      = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime  = s.drug_exposure_end_datetime,
  t.verbatim_end_date           = s.verbatim_end_date,
  t.drug_type_concept_id        = s.drug_type_concept_id,
  t.stop_reason                 = s.stop_reason,
  t.refills                     = s.refills,
  t.quantity                    = s.quantity,
  t.days_supply                 = s.days_supply,
  t.sig                         = s.sig,
  t.route_concept_id            = s.route_concept_id,
  t.lot_number                  = s.lot_number,
  t.drug_source_value           = s.drug_source_value,
  t.drug_source_concept_id      = s.drug_source_concept_id,
  t.route_source_value          = s.route_source_value,
  t.dose_unit_source_value      = s.dose_unit_source_value,
  t.person_source_value         = s.person_source_value,
  t.provider_source_value       = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value   = s.visit_detail_source_value,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.drug_exposure
# WHERE source_system = 'epic_clarity'
# LIMIT 10

In [0]:
%sql
-- Insert new mappings to source_to_drug_exposure
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
    source_system,
    drug_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.drug_exposure_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.drug_exposure
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

In [0]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.drug_exposure AS gold
MERGE INTO _exponent.omop_epic.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.drug_concept_id              = src.drug_concept_id,
  gold.drug_exposure_start_date     = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date       = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
  gold.verbatim_end_date            = src.verbatim_end_date,
  gold.drug_type_concept_id         = src.drug_type_concept_id,
  gold.stop_reason                  = src.stop_reason,
  gold.refills                      = src.refills,
  gold.quantity                     = src.quantity,
  gold.days_supply                  = src.days_supply,
  gold.sig                          = src.sig,
  gold.route_concept_id             = src.route_concept_id,
  gold.lot_number                   = src.lot_number,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.drug_source_value            = src.drug_source_value,
  gold.drug_source_concept_id       = src.drug_source_concept_id,
  gold.route_source_value           = src.route_source_value,
  gold.dose_unit_source_value       = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [0]:
# %sql
# Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.drug_exposure WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_drug_exposure WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.drug_exposure

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.drug_exposure
# WHERE drug_concept_id = 0
# LIMIT 10

In [ ]:
%sql
-- Validation queries
-- 1. Check person_id FK integrity (should be 0 orphan records)
SELECT 'DRUG_EXPOSURE.PERSON_ID FK' as check_field,
       COUNT(*) as orphan_records
FROM _exponent.omop_epic.drug_exposure de
WHERE NOT EXISTS (SELECT 1 FROM _exponent.omop_epic.person p WHERE p.person_id = de.person_id);

-- 2. Total record count
SELECT 'total_records' as check_field, COUNT(*) as cnt FROM _exponent.omop_epic.drug_exposure;